# My Itenary

A simple one-day city itinerary planner.

Enter your details and get a time schedule of places to visit (with map links) in your target city.


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

In [2]:
MODEL = "gpt-5-nano"

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found — add OPENAI_API_KEY to your .env file.")
else:
    print("API key found.")
    openai = OpenAI()

API key found.


In [3]:
system_prompt = """
You are My Itenary, a friendly and a professional one-day city itinerary planner.

Given a traveler's age, gender, country, mood, solo or group or couple or family and target city and children age if any create a realistic
single-day schedule of places to visit.

Rules:
- Use the country + city together so you pick the correct city.
- Match the mood (e.g. relaxed = fewer stops, foodie = markets and restaurants,
  adventurous = outdoor or active spots, cultural = museums and landmarks, historical = historical places , romantic = romantic places , shopping = shopping places).
- Adjust pace for age (more breaks for older travelers, livelier pace for younger).
- Cluster nearby places to minimize back-and-forth travel.
- Schedule from 09:00 to 21:00 unless told otherwise.
- Output ONLY place names with times — no long descriptions.
- Format each line as: HH:MM – Place name
- Include a short header with city and mood, then the schedule.
- Do not invent prices, tickets, only add map links.
"""

In [4]:
def build_user_prompt(age, gender, country, mood, travel_type, city, children_age=""):
    children_line = (
        f"- Children ages: {children_age}\n"
        if children_age.strip()
        else "- Children ages: none\n"
    )
    return f"""
Plan one full day in {city}, {country}.

Traveler profile:
- Age: {age}
- Gender: {gender}
- Mood: {mood}
- Trip type: {travel_type}
{children_line}
Return a time schedule from 09:00 to 21:00.
Format each stop as: HH:MM – Place name – [Google Maps link](url)
Include a short header with city, mood, and trip type, then the schedule.
"""


def generate_day_plan(age, gender, country, mood, travel_type, city, children_age=""):
    messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": build_user_prompt(
                age, gender, country, mood, travel_type, city, children_age
            ),
        },
    ]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Run the planner

**Edit the text values** in the cell below, then run that cell.

Fields: `age`, `gender`, `country`, `mood`, `travel_type` (solo / group / couple / family), `city`, and `children_age` (e.g. `5, 8` for family trips, or leave `""` if none).

In [5]:
# --- Edit your details here, then run this cell ---

age = "46"
gender = "male" # male | female
country = "Thailand"
mood = "adventurous" # relaxed | adventurous | foodie | cultural | historical | romantic | shopping
travel_type = "family"  # solo | group | couple | family
city = "Pattaya"
children_age = "20"  # e.g. "5, 8" when travel_type is family; else leave empty

# ---------------------------------------------------

print("=== My Itenary ===")
print(f"Planning a {mood} {travel_type} day in {city}, {country}...\n")

required = [age, gender, country, mood, travel_type, city]
if not all(required):
    print("Please fill in all required fields above.")
elif travel_type.strip().lower() == "family" and not children_age.strip():
    print('For family trips, set children_age (e.g. "5, 8").')
else:
    plan = generate_day_plan(
        age, gender, country, mood, travel_type, city, children_age
    )
    display(Markdown(plan))

=== My Itenary ===
Planning a adventurous family day in Pattaya, Thailand...



Pattaya — Adventurous mood — Family trip

09:00 – The Sanctuary of Truth – [Google Maps link](https://www.google.com/maps/place/The+Sanctuary+of+Truth)
11:30 – Pattaya View Point – [Google Maps link](https://www.google.com/maps/place/Pattaya+View+Point)
12:15 – Wat Phra Yai (Big Buddha) – [Google Maps link](https://www.google.com/maps/place/Wat+Phra+Yai)
13:00 – Nong Nooch Tropical Botanical Garden – [Google Maps link](https://www.google.com/maps/place/Nong+Nooch+Tropical+Botanical+Garden)
16:30 – Cartoon Network Amazone Water Park – [Google Maps link](https://www.google.com/maps/place/Cartoon+Network+Amazone+Water+Park)
18:30 – Pattaya Floating Market – [Google Maps link](https://www.google.com/maps/place/Pattaya+Floating+Market)
20:30 – Pattaya Beach – [Google Maps link](https://www.google.com/maps/place/Pattaya+Beach)